# Solutions — TPC-H Complex Queries (Hard 03)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")


## Solution 1 — Pricing Summary (TPC-H Q1 variant)

In [ ]:
result_1 = (
    lineitem
    .filter(F.col("l_shipdate") <= "1998-09-01")
    .groupBy("l_returnflag", "l_linestatus")
    .agg(
        F.round(F.sum("l_quantity"),                                                    2).alias("sum_qty"),
        F.round(F.sum("l_extendedprice"),                                               2).alias("sum_base_price"),
        F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))),           2).alias("sum_disc_price"),
        F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount")) * (1 + F.col("l_tax"))), 2).alias("sum_charge"),
        F.round(F.avg("l_quantity"),                                                    2).alias("avg_qty"),
        F.round(F.avg("l_extendedprice"),                                               2).alias("avg_price"),
        F.round(F.avg("l_discount"),                                                    4).alias("avg_disc"),
        F.count("*").alias("count_order"),
    )
    .orderBy("l_returnflag", "l_linestatus")
)


## Solution 2 — Shipping Priority (TPC-H Q3 variant)

In [ ]:
result_2 = (
    customer.filter(F.col("c_mktsegment") == "BUILDING")
    .join(orders.filter(F.col("o_orderdate") < "1995-03-15"), F.col("c_custkey") == F.col("o_custkey"))
    .join(lineitem.filter(F.col("l_shipdate") > "1995-03-15"), F.col("o_orderkey") == F.col("l_orderkey"))
    .groupBy("l_orderkey", "o_orderdate", "o_shippriority")
    .agg(F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("revenue"))
    .select("l_orderkey", "revenue", "o_orderdate", "o_shippriority")
    .orderBy(F.col("revenue").desc())
    .limit(10)
)


## Solution 3 — Order Priority Checking (TPC-H Q4 variant)

In [ ]:
late_orders = (
    lineitem
    .filter(F.col("l_receiptdate") > F.col("l_commitdate"))
    .select("l_orderkey")
    .distinct()
)

result_3 = (
    orders
    .filter(
        (F.col("o_orderdate") >= "1993-07-01") &
        (F.col("o_orderdate") < "1993-10-01")
    )
    .join(late_orders, F.col("o_orderkey") == F.col("l_orderkey"))
    .groupBy("o_orderpriority")
    .agg(F.count("*").alias("order_count"))
    .orderBy("o_orderpriority")
)


## Solution 4 — Local Supplier Volume (TPC-H Q5 variant)

In [ ]:
result_4 = (
    orders.filter((F.col("o_orderdate") >= "1994-01-01") & (F.col("o_orderdate") < "1995-01-01"))
    .join(customer,  F.col("o_custkey") == F.col("c_custkey"))
    .join(lineitem,  F.col("o_orderkey") == F.col("l_orderkey"))
    .join(supplier,  F.col("l_suppkey") == F.col("s_suppkey"))
    .join(nation.alias("cn"), F.col("c_nationkey") == F.col("cn.n_nationkey"))
    .join(nation.alias("sn"), F.col("s_nationkey") == F.col("sn.n_nationkey"))
    .filter(F.col("c_nationkey") == F.col("s_nationkey"))
    .groupBy(F.col("cn.n_name"))
    .agg(F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("revenue"))
    .withColumnRenamed("cn.n_name", "n_name")
    .toDF("n_name", "revenue")
    .orderBy(F.col("revenue").desc())
)


## Solution 5 — Revenue Change Forecast (TPC-H Q6 variant)

In [ ]:
result_5 = (
    lineitem
    .filter(
        (F.col("l_shipdate") >= "1994-01-01") & (F.col("l_shipdate") < "1995-01-01") &
        (F.col("l_discount").between(0.05, 0.07)) &
        (F.col("l_quantity") < 24)
    )
    .agg(F.round(F.sum(F.col("l_extendedprice") * F.col("l_discount")), 2).alias("revenue_change"))
)

rc = result_5.collect()[0]["revenue_change"]

## Solution 6 — Volume Shipping (TPC-H Q7 variant)

In [ ]:
nation2 = nation.alias("n2")

result_6 = (
    supplier
    .join(nation.alias("sn"), F.col("s_nationkey") == F.col("sn.n_nationkey"))
    .join(lineitem, F.col("s_suppkey") == F.col("l_suppkey"))
    .filter(
        (F.col("l_shipdate").between("1995-01-01", "1996-12-31")) &
        F.col("sn.n_name").isin("FRANCE", "GERMANY")
    )
    .join(orders, F.col("l_orderkey") == F.col("o_orderkey"))
    .join(customer, F.col("o_custkey") == F.col("c_custkey"))
    .join(nation.alias("cn"), F.col("c_nationkey") == F.col("cn.n_nationkey"))
    .filter(F.col("cn.n_name").isin("FRANCE", "GERMANY"))
    .filter(F.col("sn.n_name") != F.col("cn.n_name"))
    .groupBy(
        F.col("sn.n_name").alias("supp_nation"),
        F.col("cn.n_name").alias("cust_nation"),
        F.year("l_shipdate").alias("l_year"),
    )
    .agg(F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("revenue"))
    .orderBy("supp_nation", "cust_nation", "l_year")
)


## Solution 7 — Market Share Analysis by Region

In [ ]:
# Revenue per region per year
rev_per_region = (
    lineitem
    .join(orders,   F.col("l_orderkey") == F.col("o_orderkey"))
    .join(customer, F.col("o_custkey")  == F.col("c_custkey"))
    .join(nation,   F.col("c_nationkey") == F.col("n_nationkey"))
    .join(region,   F.col("n_regionkey") == F.col("r_regionkey"))
    .groupBy(F.col("r_name"), F.year("o_orderdate").alias("o_year"))
    .agg(F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("regional_revenue"))
)

total_per_year = (
    rev_per_region
    .groupBy("o_year")
    .agg(F.sum("regional_revenue").alias("total_revenue"))
)

result_7 = (
    rev_per_region
    .join(total_per_year, on="o_year")
    .withColumn("market_share_pct", F.round(F.col("regional_revenue") / F.col("total_revenue") * 100, 2))
    .orderBy("r_name", "o_year")
)


## Solution 8 — Supplier Reliability

In [ ]:
result_8 = (
    partsupp
    .join(supplier, F.col("ps_suppkey") == F.col("s_suppkey"))
    .join(nation,   F.col("s_nationkey") == F.col("n_nationkey"))
    .join(region,   F.col("n_regionkey") == F.col("r_regionkey"))
    .groupBy("s_name", "n_name", "r_name")
    .agg(
        F.countDistinct("ps_partkey").alias("total_parts"),
        F.round(F.avg("ps_supplycost"), 2).alias("avg_supply_cost"),
        F.sum(F.when(F.col("ps_availqty") < 100, 1).otherwise(0)).alias("low_stock_count"),
    )
    .withColumn("low_stock_pct", F.round(F.col("low_stock_count") / F.col("total_parts") * 100, 2))
    .orderBy(F.col("low_stock_pct").desc())
)
